In [22]:
import plotly.io as pio
pio.renderers.default = "notebook"

In [23]:
# === Step 1: Import Required Libraries and Set Plot Style ===
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected"
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", None)

# Color palette that is easy to distinguish, including for people with color vision deficiency
OKABE_ITO = ["#0072B2", "#E69F00", "#009E73", "#D55E00",
             "#56B4E9", "#CC79A7", "#F0E442", "#000000"]
HIGHLIGHT = "#0072B2"   # Highlight important information
GREY      = "#BDBDBD"   # Use grey for supporting information

def style_fig(fig, title, subtitle=None):
    """Function to apply the same design style to all graphs."""
    full = f"<b>{title}</b>"
    if subtitle:
        full += f"<br><span style='font-size:13px;color:#666'>{subtitle}</span>"
    fig.update_layout(
        title=dict(text=full, x=0.01, xanchor="left"),
        template="plotly_white",
        font=dict(family="Arial", size=13, color="#222"),
        margin=dict(t=95, r=30, b=60, l=80),
        plot_bgcolor="white",
    )
    fig.update_xaxes(showgrid=False, zeroline=False, showline=True, linecolor="#ccc")
    fig.update_yaxes(showgrid=True, gridcolor="#eee", zeroline=False)
    return fig

In [24]:
# === Step 2: Load the IMDb Movie Dataset ===
df = pd.read_csv("dataset/imdb_movies.csv")   # Load the dataset from the dataset folder
df.columns = df.columns.str.strip()  # Remove extra spaces from column names
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()  # Display the first five rows of the dataset

Shape: (10178, 12)
Columns: ['names', 'date_x', 'score', 'genre', 'overview', 'crew', 'orig_title', 'status', 'orig_lang', 'budget_x', 'revenue', 'country']


,names,date_x,score,genre,overview,crew,orig_title,status,orig_lang,budget_x,revenue,country
0,Creed III,03-02-2023,73,"Drama, Action","After dominating the boxing world, Adonis Cree...","Michael B. Jordan, Adonis Creed, Tessa Thompso...",Creed III,Released,English,75000000.0,2.716167e+08,AU
1,Avatar: The Way of Water,12/15/2022,78,"Science Fiction, Adventure, Action",Set more than a decade after the events of the...,"Sam Worthington, Jake Sully, Zoe Saldaña, Neyt...",Avatar: The Way of Water,Released,English,460000000.0,2.316795e+09,AU
2,The Super Mario Bros. Movie,04-05-2023,76,"Animation, Adventure, Family, Fantasy, Comedy","While working underground to fix a water main,...","Chris Pratt, Mario (voice), Anya Taylor-Joy, P...",The Super Mario Bros. Movie,Released,English,100000000.0,7.244590e+08,AU
3,Mummies,01-05-2023,70,"Animation, Comedy, Family, Adventure, Fantasy","Through a series of unfortunate events, three ...","Óscar Barberán, Thut (voice), Ana Esther Albor...",Momias,Released,"Spanish, Castilian",12300000.0,3.420000e+07,AU
4,Supercell,03/17/2023,61,Action,Good-hearted teenager William always lived in ...,"Skeet Ulrich, Roy Cameron, Anne Heche, Dr Quin...",Supercell,Released,English,77000000.0,3.409420e+08,US


In [25]:
# === Step 3: Clean and Prepare the Dataset ===

# Rename columns to make them easier to understand
df = df.rename(columns={
    "names": "title",
    "date_x": "release_date",
    "score": "rating",
    "orig_lang": "language",
    "budget_x": "budget",
})

# Extract year and month from the release date
df["release_date"] = pd.to_datetime(df["release_date"], format="mixed", errors="coerce")
df["year"]  = df["release_date"].dt.year
df["month"] = df["release_date"].dt.month

# Convert rating, budget and revenue columns to numeric values
for c in ["rating", "budget", "revenue"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Replace zero values with missing values because they usually represent unavailable data
df.loc[df["budget"]  == 0, "budget"]  = np.nan
df.loc[df["revenue"] == 0, "revenue"] = np.nan

# Calculate profit and return on investment (ROI)
df["profit"] = df["revenue"] - df["budget"]
df["roi"]    = df["revenue"] / df["budget"]      # > 1 = made its money back

# Clean text columns by removing extra spaces
for c in ["genre", "language", "country", "status"]:
    if c in df.columns:
        df[c] = df[c].astype("string").str.strip()

df[["title","year","rating","budget","revenue","roi","genre","language","country"]].head()

,title,year,rating,budget,revenue,roi,genre,language,country
0,Creed III,2023,73,75000000.0,2.716167e+08,3.621556,"Drama, Action",English,AU
1,Avatar: The Way of Water,2022,78,460000000.0,2.316795e+09,5.036511,"Science Fiction, Adventure, Action",English,AU
2,The Super Mario Bros. Movie,2023,76,100000000.0,7.244590e+08,7.244590,"Animation, Adventure, Family, Fantasy, Comedy",English,AU
3,Mummies,2023,70,12300000.0,3.420000e+07,2.780488,"Animation, Comedy, Family, Adventure, Fantasy","Spanish, Castilian",AU
4,Supercell,2023,61,77000000.0,3.409420e+08,4.427818,Action,English,US


In [26]:
# === Step 4: Prepare Genre Data for Analysis ===

# Split movies with multiple genres into separate rows
genre_df = df.assign(genre=df["genre"].str.split(",")).explode("genre")
genre_df["genre"] = genre_df["genre"].str.strip()
# Remove empty or missing genre values
genre_df = genre_df[genre_df["genre"].notna() & (genre_df["genre"] != "")]
# Display the most common genres
genre_df["genre"].value_counts().head(15)

genre
Drama              3812
Comedy             2943
Action             2752
Thriller           2605
Adventure          1890
Romance            1576
Horror             1554
Animation          1468
Family             1407
Fantasy            1382
Crime              1272
Science Fiction    1261
Mystery             862
History             422
War                 282
Name: count, dtype: int64

In [27]:
# === Step 5: Perform Initial Data Exploration ===
print("Movies:", len(df))   # Display basic information about the dataset
print("Year range:", int(df["year"].min()), "-", int(df["year"].max()))
df[["rating","budget","revenue","roi"]].describe() # Display summary statistics for important numerical columns

Movies: 10178
Year range: 1903 - 2023


,rating,budget,revenue,roi
count,10178.000000,1.017800e+04,1.010500e+04,1.010500e+04
mean,63.497052,6.488238e+07,2.549688e+08,2.985619e+04
std,13.537012,5.707565e+07,2.779522e+08,2.576620e+06
min,0.000000,1.000000e+00,1.000000e+00,1.271858e-06
25%,59.000000,1.500000e+07,2.995823e+07,1.439412e+00
50%,65.000000,5.000000e+07,1.579565e+08,3.318530e+00
75%,71.000000,1.050000e+08,4.216748e+08,5.878644e+00
max,100.000000,4.600000e+08,2.923706e+09,2.577204e+08


In [28]:
# === Analysis 1: Does a Bigger Budget Lead to Higher Revenue? ===
q1 = df.dropna(subset=["budget", "revenue"]) # Remove movies with missing budget or revenue values
# Create a scatter plot to compare budget and revenue
fig = px.scatter(q1, x="budget", y="revenue", color="rating",
                 color_continuous_scale="Viridis", hover_name="title")
fig.update_traces(marker=dict(size=6, opacity=0.55)) # Adjust marker size and transparency for better visibility
fig.update_xaxes(title="Budget (USD)")
fig.update_yaxes(title="Revenue (USD)")
style_fig(fig, "Bigger budgets tend to earn more — but the payoff is far from guaranteed",
          "Each dot is a film · colour = rating")
fig.write_image("images/q1.png", width=1000, height=600, scale=2)
fig.show()

### Conclusion

The scatter plot shows an overall positive relationship between production budget and box-office revenue. While many high-budget movies generate higher revenue, there are several exceptions where expensive movies perform poorly and lower-budget movies achieve strong financial success. This suggests that budget is an important factor but not the only determinant of commercial success.

In [29]:
# === Analysis 2: Which Genres Have the Best Return on Investment? ===
# Calculate the median ROI for each genre
roi_by_genre = (genre_df.dropna(subset=["roi"])
                .groupby("genre")["roi"].median()
                .sort_values(ascending=False).head(12).reset_index())
colors = [HIGHLIGHT if i == 0 else GREY for i in range(len(roi_by_genre))]
fig = go.Figure(go.Bar(x=roi_by_genre["roi"], y=roi_by_genre["genre"],
                       orientation="h", marker_color=colors,
                       text=roi_by_genre["roi"].round(1), textposition="outside"))
fig.update_yaxes(autorange="reversed", title="")
fig.update_xaxes(title="Median return on budget (revenue ÷ budget)")
style_fig(fig, f"{roi_by_genre['genre'].iloc[0]} films give the strongest return on budget",
          "Median revenue per $1 of budget, by genre")
fig.write_image("images/q2.png", width=1000, height=600, scale=2)
fig.show()

### Conclusion

Among all genres, documentary movies have the highest median return on investment (ROI) in this dataset. This indicates that documentaries are generally produced with lower budgets while still generating strong returns relative to their production costs. Other genres show lower median ROI despite having larger budgets.

In [30]:
# === Analysis 3: How Have Movie Ratings Changed Across Decades? ===
top_genres = genre_df["genre"].value_counts().head(4).index.tolist()
sub = genre_df[genre_df["genre"].isin(top_genres) & genre_df["year"].notna()].copy()
sub["decade"] = (sub["year"] // 10) * 10
trend = sub.groupby(["decade", "genre"], observed=True)["rating"].mean().reset_index()
fig = px.line(trend, x="decade", y="rating", color="genre",
              color_discrete_sequence=OKABE_ITO, markers=True)
fig.update_xaxes(title="Decade"); fig.update_yaxes(title="Average rating")
style_fig(fig, "Average ratings have drifted downward since the mid-century",
          "Mean rating per decade for the four most common genres")
fig.show()

### Conclusion

The average IMDb ratings for the four most common genres show a gradual decline over the decades. Although the decrease is not dramatic, movies released in recent decades generally receive slightly lower average ratings than those released during the mid-20th century.

In [31]:
# === Analysis 4: Comparing English and Non-English Movies ===
df["lang_group"] = np.where(df["language"].str.lower() == "english", "English", "Non-English")
g = df.groupby("lang_group").agg(avg_rating=("rating","mean"),
                                 avg_budget=("budget","mean")).reset_index()
fig = make_subplots(rows=1, cols=2, subplot_titles=("Average rating", "Average budget (USD)"))
fig.add_trace(go.Bar(x=g["lang_group"], y=g["avg_rating"],
                     marker_color=[HIGHLIGHT, GREY]), row=1, col=1)
fig.add_trace(go.Bar(x=g["lang_group"], y=g["avg_budget"],
                     marker_color=[HIGHLIGHT, GREY]), row=1, col=2)
fig.update_layout(showlegend=False)
style_fig(fig, "Non-English films cost more on average, yet score about the same")
fig.show()

### Conclusion

The comparison shows that non-English movies have a higher average production budget in this dataset, while the average IMDb ratings of English and non-English movies remain very similar. This suggests that higher production spending does not necessarily lead to better audience ratings.

In [32]:
# === Analysis 5: Which Countries Perform Better Than Expected? ===
c = (df.groupby("country")
       .agg(n=("title","size"), avg_rating=("rating","mean"), avg_rev=("revenue","mean"))
       .reset_index())
c = c[c["n"] >= 30].copy()
c["avg_rev"] = c["avg_rev"].fillna(0)
fig = px.scatter(c, x="n", y="avg_rating", size="avg_rev", hover_name="country",
                 text="country", size_max=40, color_discrete_sequence=[HIGHLIGHT])
fig.update_traces(textposition="top center", marker=dict(opacity=0.6))
fig.update_xaxes(title="Number of films produced")
fig.update_yaxes(title="Average rating")
style_fig(fig, "A few smaller producers rival the giants on average rating",
          "Countries with 30+ films · bubble size = average revenue")
fig.show()

### Conclusion

Although some countries produce a large number of movies, several countries with fewer productions achieve comparable or even higher average IMDb ratings. This suggests that producing more films does not automatically result in higher-quality movies.

In [33]:
# === Analysis 6: Finding the Most Profitable Budget Range ===
bins   = [0, 1e6, 1e7, 5e7, 1e8, 2e8, np.inf]
labels = ["<1M", "1–10M", "10–50M", "50–100M", "100–200M", ">200M"]
df["budget_tier"] = pd.cut(df["budget"], bins=bins, labels=labels)
t = (df.dropna(subset=["roi"])
       .groupby("budget_tier", observed=True)["roi"].median().reset_index())
peak = t["roi"].idxmax()
colors = [HIGHLIGHT if i == peak else GREY for i in range(len(t))]
fig = go.Figure(go.Bar(x=t["budget_tier"].astype(str), y=t["roi"],
                       marker_color=colors, text=t["roi"].round(1), textposition="outside"))
fig.update_xaxes(title="Budget tier (USD)"); fig.update_yaxes(title="Median ROI")
style_fig(fig, "Lower-budget films tend to return the most per dollar spent",
          "Median revenue ÷ budget within each budget tier")
fig.write_image("images/q6.png", width=1000, height=600, scale=2)
fig.show()

### Conclusion

The analysis indicates that lower-budget movies generally achieve higher median return on investment than high-budget productions. While large-budget movies may earn higher total revenue, they often require significantly greater investment, reducing their overall efficiency.

In [34]:
# === Analysis 7: Comparing Average Budget and Revenue by Genre ===
g = (genre_df.groupby("genre")
       .agg(avg_budget=("budget","mean"), avg_revenue=("revenue","mean"), n=("title","size"))
       .reset_index())
g = g[g["n"] >= 20]
fig = px.scatter(g, x="avg_budget", y="avg_revenue", size="n", text="genre",
                 hover_name="genre", size_max=45, color_discrete_sequence=[HIGHLIGHT])
fig.update_traces(textposition="top center", marker=dict(opacity=0.65))
fig.update_xaxes(title="Average budget (USD)"); fig.update_yaxes(title="Average revenue (USD)")
style_fig(fig, "Big-budget genres also bring in the biggest revenue",
          "Each dot is a genre · bubble size = number of films")
fig.show()

### Conclusion

Genres with higher average production budgets also tend to generate higher average box-office revenue. However, the relationship is not perfectly proportional, indicating that increased spending alone does not guarantee greater profitability.

In [35]:
# === Analysis 8: How Movie Genres Have Changed Over Time ===
s = genre_df.dropna(subset=["year"]).copy()
s["decade"] = (s["year"] // 10) * 10
top8 = s["genre"].value_counts().head(8).index
s = s[s["genre"].isin(top8)]
counts = s.groupby(["decade","genre"], observed=True).size().reset_index(name="n")
counts["share"] = counts["n"] / counts.groupby("decade")["n"].transform("sum")
fig = px.area(counts, x="decade", y="share", color="genre",
              color_discrete_sequence=OKABE_ITO)
fig.update_yaxes(title="Share of releases", tickformat=".0%")
fig.update_xaxes(title="Decade")
style_fig(fig, "The genre balance of releases has shifted over the decades",
          "Share of films by genre within each decade")
fig.write_image("images/q8.png", width=1000, height=600, scale=2)
fig.show()

### Conclusion

The proportion of movie genres has changed considerably across different decades. Some genres have become more common while others have declined, reflecting changes in audience preferences and trends within the film industry. Movies used to be mostly Adventure and Drama (about half each in the early days). Now it's spread out a lot more across comedy, thriller, horror, action and the rest. The industry got more varied.

In [36]:
# === Analysis 9: Is Movie Rating Related to Revenue? ===
q9 = df.dropna(subset=["rating","revenue"]).copy()
q9["group"] = np.where(q9["rating"] >= 80, "Highly rated (80+)", "Rest")
fig = px.scatter(q9, x="rating", y="revenue", color="group", hover_name="title",
                 color_discrete_map={"Highly rated (80+)": HIGHLIGHT, "Rest": GREY})
fig.update_traces(marker=dict(size=6, opacity=0.5))
fig.update_xaxes(title="Rating"); fig.update_yaxes(title="Revenue (USD)")
style_fig(fig, "High ratings don't guarantee high box office", "Each dot is a film")
fig.write_image("images/q9.png", width=1000, height=600, scale=2)
fig.show()

### Conclusion

The scatter plot shows that highly rated movies do not always achieve the highest box-office revenue. While some critically acclaimed movies are commercially successful, many moderate-rated movies also generate significant revenue, indicating that audience ratings alone do not determine financial success.

In [37]:
# === Analysis 10: Which Release Month Generates the Highest Revenue? ===
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
m = (df.dropna(subset=["month","revenue"])
       .groupby("month")["revenue"].median().reset_index())
m["month_name"] = m["month"].astype(int).apply(lambda x: month_names[x-1])
peak = m["revenue"].idxmax()
colors = [HIGHLIGHT if i == peak else GREY for i in range(len(m))]
fig = go.Figure(go.Bar(x=m["month_name"], y=m["revenue"], marker_color=colors))
fig.update_xaxes(title="Release month"); fig.update_yaxes(title="Median revenue (USD)")
style_fig(fig, "Films released in peak season out-earn the rest",
          "Median box-office revenue by release month")
fig.write_image("images/q10.png", width=1000, height=600, scale=2)
fig.show()

### Conclusion

Median box-office revenue varies across different release months, with certain months consistently generating higher revenue than others. This suggests that choosing the right release period can positively influence a movie's commercial performance.

In [38]:
# === Analysis 11: Profitability Comparison Across Countries ===
top_countries = df["country"].value_counts().head(10).index
cc = (df[df["country"].isin(top_countries)].dropna(subset=["roi"])
        .groupby("country")["roi"].median().sort_values(ascending=False).reset_index())
colors = [HIGHLIGHT if i == 0 else GREY for i in range(len(cc))]
fig = go.Figure(go.Bar(x=cc["country"], y=cc["roi"], marker_color=colors,
                       text=cc["roi"].round(1), textposition="outside"))
fig.update_xaxes(title="Country"); fig.update_yaxes(title="Median ROI")
style_fig(fig, "Profitability varies widely across the biggest film-producing countries",
          "Median revenue ÷ budget · top 10 countries by film count")
fig.show()

### Conclusion

The median return on investment differs considerably among the top movie-producing countries. Some countries achieve stronger profitability than others, indicating differences in production costs, market size, and overall filmmaking strategies.

In [39]:
# === Analysis 12: Does Budget Affect the Rating–Revenue Relationship? ===
q12 = df.dropna(subset=["rating","revenue","budget_tier"])
fig = px.scatter(q12, x="rating", y="revenue", color="budget_tier",
                 hover_name="title", color_discrete_sequence=OKABE_ITO,
                 category_orders={"budget_tier": labels})
fig.update_traces(marker=dict(size=6, opacity=0.5))
fig.update_xaxes(title="Rating"); fig.update_yaxes(title="Revenue (USD)")
style_fig(fig, "Budget shapes the rating–revenue link",
          "Each dot is a film · colour = budget tier")
fig.write_image("images/q12.png", width=1000, height=600, scale=2)
fig.show()

### Conclusion

The relationship between IMDb ratings and revenue varies across different budget categories. Higher-budget movies generally have greater revenue potential, but high ratings alone are not sufficient to ensure commercial success.

# Overall Conclusion

This project analyzed more than 10,000 IMDb movies to understand how production budget, genre, language, release timing, country, and audience ratings influence commercial success. The analysis shows that production budget is the strongest factor shaping movie performance, but its impact depends on how success is measured. Lower-budget films generally achieve the highest return on investment, making them the most financially efficient, while higher-budget films generate the greatest total revenue and are the only group where stronger audience ratings are more consistently associated with higher earnings. Audience ratings alone are weak predictors of box-office success, indicating that critical appreciation does not necessarily translate into commercial performance. The analysis also revealed that genre, release timing, and production country influence profitability, while the distribution of movie genres has evolved over time. Overall, the findings demonstrate that movie success results from the interaction of multiple factors rather than any single characteristic, highlighting the value of data visualization in uncovering meaningful patterns within complex real-world datasets.